In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

In [2]:
from datetime import datetime, timedelta

# Hourly data is limited to the last 1 year
start_date = "2025-01-01"
end_date = "2026-06-01"
BankNifty_data = yf.download('^NSEBANK', start=start_date, end=end_date, interval='1h')
display(BankNifty_data.head())

/tmp/ipykernel_6105/4171643757.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  BankNifty_data = yf.download('^NSEBANK', start=start_date, end=end_date, interval='1h')
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK,^NSEBANK
Datetime,,,,,
2025-01-01 03:45:00+00:00,50790.250000,50969.148438,50486.601562,50846.949219,0
2025-01-01 04:45:00+00:00,51103.500000,51161.199219,50765.148438,50790.050781,0
2025-01-01 05:45:00+00:00,51059.800781,51167.398438,50971.101562,51108.101562,0
2025-01-01 06:45:00+00:00,51012.750000,51153.750000,51001.601562,51059.550781,0
2025-01-01 07:45:00+00:00,51232.601562,51321.101562,51008.300781,51013.398438,0


In [3]:
# Create a copy and flatten the MultiIndex columns from yfinance
clean_df = BankNifty_data.copy()
clean_df.columns = clean_df.columns.get_level_values(0)

# Reset index to turn 'Datetime' into a column
clean_df = clean_df.reset_index()

# Create separate Date and Time columns
clean_df['Date'] = clean_df['Datetime'].dt.date
clean_df['Time'] = clean_df['Datetime'].dt.time
final_columns = ['Date', 'Time', 'Open', 'High', 'Low', 'Close', 'Volume']
final_df = clean_df[final_columns]

display(final_df.head())

Price,Date,Time,Open,High,Low,Close,Volume
0,2025-01-01,03:45:00,50846.949219,50969.148438,50486.601562,50790.250000,0
1,2025-01-01,04:45:00,50790.050781,51161.199219,50765.148438,51103.500000,0
2,2025-01-01,05:45:00,51108.101562,51167.398438,50971.101562,51059.800781,0
3,2025-01-01,06:45:00,51059.550781,51153.750000,51001.601562,51012.750000,0
4,2025-01-01,07:45:00,51013.398438,51321.101562,51008.300781,51232.601562,0


In [4]:
final_df['hourly_return'] = final_df['Close'].pct_change()
final_df['hourly_return'].fillna(0, inplace=True)
final_df.head()

/tmp/ipykernel_6105/450399559.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  final_df['hourly_return'].fillna(0, inplace=True)


Price,Date,Time,Open,High,Low,Close,Volume,hourly_return
0,2025-01-01,03:45:00,50846.949219,50969.148438,50486.601562,50790.250000,0,NaN
1,2025-01-01,04:45:00,50790.050781,51161.199219,50765.148438,51103.500000,0,0.006168
2,2025-01-01,05:45:00,51108.101562,51167.398438,50971.101562,51059.800781,0,-0.000855
3,2025-01-01,06:45:00,51059.550781,51153.750000,51001.601562,51012.750000,0,-0.000921
4,2025-01-01,07:45:00,51013.398438,51321.101562,51008.300781,51232.601562,0,0.004310


In [5]:
def average_true_range(close_prices, high_prices, low_prices, period):
    true_ranges = np.maximum(high_prices - low_prices, np.abs(high_prices - close_prices.shift(1)), np.abs(low_prices - close_prices.shift(1)))
    atr = true_ranges.ewm(alpha = 1/(period), adjust=False).mean()
    atr = atr/close_prices * 100

    return atr

In [6]:
def calculate_dynamic_smoothing_factor(df, atr_period=14):
    """
    Calculates a dynamic smoothing factor based on ATR.
    Uses Min-Max scaling to normalize ATR values to a [0,1] range.

    Args:
        df (pd.DataFrame): DataFrame with 'Open', 'High', 'Low', 'Close' prices.
        atr_period (int): Period for ATR calculation.

    Returns:
        pd.Series: Series of dynamic smoothing factors.
    """
    # Ensure the DataFrame is sorted by date for correct calculations
    df = df.sort_values(by='Date')

    # Calculate ATR
    df['ATR'] = average_true_range(df['Close'], df['High'], df['Low'], atr_period)

    # Calculate min and max of ATR for Min-Max scaling
    min_atr = df['ATR'].min()
    max_atr = df['ATR'].max()
    # Apply Min-Max scaling to normalize ATR to [0, 1]
    normalized_atr = (df['ATR'] - min_atr) / (max_atr - min_atr)
    # Clip to ensure smoothing factor is within valid bounds (e.g., 0.01 to 1)
    dynamic_smoothing_factor = normalized_atr.clip(upper=1.0)

    return pd.Series(dynamic_smoothing_factor, index=df.index)

Now, let's apply this function to `final_df` and display the first few calculated dynamic smoothing factors.

In [7]:
# Calculate dynamic smoothing factors for final_df
final_df['dynamic_smoothing_factor'] = calculate_dynamic_smoothing_factor(final_df.copy())

# Display the DataFrame with the new smoothing factors
display(final_df[['Date', 'Close', 'dynamic_smoothing_factor']].head())

Price,Date,Close,dynamic_smoothing_factor
0,2025-01-01,50790.250000,NaN
1,2025-01-01,51103.500000,0.529228
2,2025-01-01,51059.800781,0.504048
3,2025-01-01,51012.750000,0.475007
4,2025-01-01,51232.601562,0.465404


In [8]:
def calculate_adaptive_ema(data, dynamic_smoothing_factors):
    """
    Calculates an Adaptive Exponential Moving Average (AEMA).

    Args:
        data (pd.Series): The time series data to smooth (e.g., 'Close' prices).
        dynamic_smoothing_factors (pd.Series): A series of smoothing factors (alpha).

    Returns:
        pd.Series: The calculated Adaptive EMA.
    """
    aema = pd.Series(index=data.index, dtype=float)

    aema.iloc[0] = data.iloc[0] # Initialize the first AEMA value

    for i in range(1, len(data)):
        alpha = dynamic_smoothing_factors.iloc[i]
        aema.iloc[i] = (alpha * data.iloc[i]) + ((1 - alpha) * aema.iloc[i-1])

    return aema


Now, let's apply the `calculate_adaptive_ema` function to the 'Close' price using the `dynamic_smoothing_factor` we computed.

In [9]:
# Calculate the Adaptive EMA
final_df['AEMA'] = calculate_adaptive_ema(final_df['Close'], final_df['dynamic_smoothing_factor'])

# Display the DataFrame with the new AEMA column
display(final_df[['Date', 'Close', 'dynamic_smoothing_factor', 'AEMA']].head(10))


Price,Date,Close,dynamic_smoothing_factor,AEMA
0,2025-01-01,50790.250000,NaN,50790.250000
1,2025-01-01,51103.500000,0.529228,50956.030720
2,2025-01-01,51059.800781,0.504048,51008.335819
3,2025-01-01,51012.750000,0.475007,51010.432585
4,2025-01-01,51232.601562,0.465404,51113.830847
5,2025-01-01,51047.601562,0.453219,51083.814466
6,2025-01-01,51096.050781,0.418810,51088.939154
7,2025-01-02,51109.800781,0.372767,51096.715680
8,2025-01-02,51275.851562,0.391606,51166.866371
9,2025-01-02,51126.601562,0.382785,51151.453605


To visualize the Adaptive EMA, let's plot it alongside the 'Close' price.

In [10]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=final_df['Date'],
    y=final_df['Close'],
    mode='lines',
    name='Close Price',
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=final_df['Date'],
    y=final_df['AEMA'],
    mode='lines',
    name='Adaptive EMA',
    line=dict(color='red', width=2)
))

fig.update_layout(title='BankNifty Close Price with Adaptive EMA',
                  xaxis_title='Date',
                  yaxis_title='Price',
                  xaxis_rangeslider_visible=True)

fig.show()


### Developing a momentum based strategy using AEMA

Now, let's generate the trading signals. We'll use a simple crossover strategy: buy when the Close price crosses above the AEMA, and sell when it crosses below. We will only consider these signals when the market is classified as 'Trending'.

In [12]:
!pip install pandas --upgrade

In [24]:
# Generate trading signals
final_df['signal'] = 0

# Buy only if Close > AEMA which represents bullish momentum signal
final_df.loc[(final_df['Close'] > final_df['AEMA']), 'signal'] = 1

# Sell only if Close < AEMA which represents bearish momentum signal
final_df.loc[(final_df['Close'] < final_df['AEMA']), 'signal'] = -1

# Calculate positions: 1 for long, -1 for short, 0 for cash, the shift(1) ensures we trade on the next hour's open
final_df['position'] = final_df['signal'].shift(1).fillna(0)

display(final_df[['Date', 'Time', 'Close', 'AEMA', 'signal', 'position']].head())

Price,Date,Time,Close,AEMA,signal,position
0,2025-01-01,03:45:00,50790.250000,50790.250000,0,0.0
1,2025-01-01,04:45:00,51103.500000,50956.030720,1,0.0
2,2025-01-01,05:45:00,51059.800781,51008.335819,1,1.0
3,2025-01-01,06:45:00,51012.750000,51010.432585,1,1.0
4,2025-01-01,07:45:00,51232.601562,51113.830847,1,1.0


Next, we will backtest the strategy by calculating the hourly returns based on our positions and then visualize the cumulative returns (equity curve) of the strategy.

In [14]:
# Recalculate strategy returns based on updated positions
final_df['strategy_return'] = final_df['hourly_return'] * final_df['position']
final_df['cumulative_strategy_return'] = (1 + final_df['strategy_return']).cumprod() - 1

# Plot the cumulative returns
fig = go.Figure()
fig.add_trace(go.Scatter(x=final_df['Date'], y=final_df['cumulative_strategy_return'], mode='lines', name='Strategy Cumulative Return', line=dict(color='green')))
fig.update_layout(title='Updated Strategy Cumulative Returns (Gross)', xaxis_title='Date', yaxis_title='Cumulative Return')
fig.show()

### Performance Metrics
**Detailed Trade Statistics & Information Ratio**
We will now calculate trade-specific metrics like the hit ratio and number of trades, as well as the Information Ratio relative to the benchmark.

In [15]:
# Assuming 6.5 trading hours per day and 252 trading days per year for annualization
TRADING_HOURS_PER_YEAR = 6.5 * 252

# Calculate Annualized Return
# Use the product of (1 + daily_return) for compounding, then annualize based on the number of data points
# A more robust way to annualize is to get the mean hourly return and raise it to the power of trading hours per year
annualized_return = (1 + final_df['strategy_return']).mean()**TRADING_HOURS_PER_YEAR - 1

# Calculate Max Drawdown
cumulative_returns = (1 + final_df['strategy_return']).cumprod()
peak = cumulative_returns.expanding(min_periods=1).max()
drawdown = (cumulative_returns / peak) - 1
max_drawdown = drawdown.min()

# Calculate Annualized Standard Deviation of Returns
annualized_std_dev = final_df['strategy_return'].std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Calculate Sharpe Ratio (assuming risk-free rate = 0 for simplicity)
sharpe_ratio = annualized_return / annualized_std_dev

# Calculate Downside Deviation for Sortino Ratio
downside_returns = final_df[final_df['strategy_return'] < 0]['strategy_return']
annualized_downside_std_dev = downside_returns.std() * np.sqrt(TRADING_HOURS_PER_YEAR)

# Calculate Sortino Ratio (assuming risk-free rate = 0)
sortino_ratio = annualized_return / annualized_downside_std_dev if annualized_downside_std_dev != 0 else np.nan

# Calculate Calmar Ratio
calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else np.nan

final_df['trade_id'] = (final_df['position'] != final_df['position'].shift()).cumsum()

# Filter only non-zero position periods to analyze actual trades
trades = final_df[final_df['position'] != 0].groupby('trade_id').agg({
    'strategy_return': lambda x: (1 + x).prod() - 1,
    'Date': 'first'
})

num_trades = len(trades)
pos_trades = trades[trades['strategy_return'] > 0]
neg_trades = trades[trades['strategy_return'] <= 0]

num_pos_trades = len(pos_trades)
num_neg_trades = len(neg_trades)
hit_ratio = num_pos_trades / num_trades if num_trades > 0 else 0

# Information Ratio Calculation
benchmark_return = final_df['hourly_return']
active_return = final_df['strategy_return'] - benchmark_return
tracking_error = active_return.std() * np.sqrt(TRADING_HOURS_PER_YEAR)
annualized_active_return = active_return.mean() * TRADING_HOURS_PER_YEAR
information_ratio = annualized_active_return / tracking_error if tracking_error != 0 else np.nan

# Display Results
print(f"--- Trade Statistics ---")
print(f"Total Number of Trades: {num_trades}")
print(f"Number of Positive Trades: {num_pos_trades}")
print(f"Number of Negative Trades: {num_neg_trades}")
print(f"Hit Ratio: {hit_ratio:.2%}")
print(f"Information Ratio: {information_ratio:.2f}")

print(f"Annualized Return: {annualized_return:.2%}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Sortino Ratio: {sortino_ratio:.2f}")
print(f"Calmar Ratio: {calmar_ratio:.2f}")


--- Trade Statistics ---
Total Number of Trades: 439
Number of Positive Trades: 150
Number of Negative Trades: 289
Hit Ratio: 34.17%
Information Ratio: 1.36
Annualized Return: 45.61%
Max Drawdown: -8.82%
Sharpe Ratio: 2.94
Sortino Ratio: 4.51
Calmar Ratio: 5.17


### Cost Modeling
Real-world trading involves costs. We will apply a fixed cost (brokerage + slippage) per trade (when the position changes). We now re-calculate all key ratios using net_strategy_return to see the actual impact of friction on the AEMA strategy.

In [17]:
COST_PER_TRADE = 0.0005 # 0.05% per side

# Identify trade execution points (entry/exit/flip)
final_df['trade_execution'] = (final_df['position'] != final_df['position'].shift(1)).astype(int)

# Apply costs to the returns
final_df['transaction_costs'] = final_df['trade_execution'] * COST_PER_TRADE
final_df['net_strategy_return'] = final_df['strategy_return'] - final_df['transaction_costs']
final_df['cumulative_net_return'] = (1 + final_df['net_strategy_return']).cumprod() - 1

print(f"Final Cumulative Net Return: {final_df['cumulative_net_return'].iloc[-1]:.2%}")

Final Cumulative Net Return: 37.26%


In [18]:
# Annualized Net Return
annualized_net_return = (1 + final_df['net_strategy_return']).mean()**TRADING_HOURS_PER_YEAR - 1

# Max Drawdown (Net)
cumulative_net_returns = (1 + final_df['net_strategy_return']).cumprod()
net_peak = cumulative_net_returns.expanding(min_periods=1).max()
net_drawdown = (cumulative_net_returns / net_peak) - 1
max_drawdown_net = net_drawdown.min()

# Ratios (Net)
annualized_net_std = final_df['net_strategy_return'].std() * np.sqrt(TRADING_HOURS_PER_YEAR)
sharpe_ratio_net = annualized_net_return / annualized_net_std if annualized_net_std != 0 else np.nan

downside_net = final_df[final_df['net_strategy_return'] < 0]['net_strategy_return']
annualized_downside_net_std = downside_net.std() * np.sqrt(TRADING_HOURS_PER_YEAR)
sortino_ratio_net = annualized_net_return / annualized_downside_net_std if annualized_downside_net_std != 0 else np.nan

calmar_ratio_net = annualized_net_return / abs(max_drawdown_net) if max_drawdown_net != 0 else np.nan

# Information Ratio (Net)
active_return_net = final_df['net_strategy_return'] - final_df['hourly_return']
tracking_error_net = active_return_net.std() * np.sqrt(TRADING_HOURS_PER_YEAR)
ann_active_return_net = active_return_net.mean() * TRADING_HOURS_PER_YEAR
information_ratio_net = ann_active_return_net / tracking_error_net if tracking_error_net != 0 else np.nan

# Updated Trade Stats (Net)
net_trades = final_df[final_df['position'] != 0].groupby('trade_id').agg({
    'net_strategy_return': lambda x: (1 + x).prod() - 1
})
num_pos_trades_net = len(net_trades[net_trades['net_strategy_return'] > 0])
hit_ratio_net = num_pos_trades_net / len(net_trades) if len(net_trades) > 0 else 0

# Display Summary Comparison
comparison_metrics = {
    'Metric': ['Annualized Return', 'Max Drawdown', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Hit Ratio', 'Information Ratio'],
    'Gross (No Cost)': [f"{annualized_return:.2%}", f"{max_drawdown:.2%}", f"{sharpe_ratio:.2f}", f"{sortino_ratio:.2f}", f"{calmar_ratio:.2f}", f"{hit_ratio:.2%}", f"{information_ratio:.2f}"],
    'Net (Post-Cost)': [f"{annualized_net_return:.2%}", f"{max_drawdown_net:.2%}", f"{sharpe_ratio_net:.2f}", f"{sortino_ratio_net:.2f}", f"{calmar_ratio_net:.2f}", f"{hit_ratio_net:.2%}", f"{information_ratio_net:.2f}"]
}

display(pd.DataFrame(comparison_metrics))

,Metric,Gross (No Cost),Net (Post-Cost)
0,Annualized Return,45.61%,25.44%
1,Max Drawdown,-8.82%,-11.60%
2,Sharpe Ratio,2.94,1.64
3,Sortino Ratio,4.51,2.54
4,Calmar Ratio,5.17,2.19
5,Hit Ratio,34.17%,32.12%
6,Information Ratio,1.36,0.72


### Walk-Forward Analysis
We will split the data into 3 windows. For each window, we 'train' the ATR threshold on the first 70% of data and 'test' it on the remaining 30%.

In [22]:
def run_backtest(data_slice, threshold_val):
    # Apply threshold and generate signals
    data_slice = data_slice.copy()
    data_slice['market_condition'] = np.where(data_slice['dynamic_smoothing_factor'] > threshold_val, 'Trending', 'Flat')
    data_slice['sig'] = np.where(data_slice['Close'] > data_slice['AEMA'], 1, -1)
    data_slice['pos'] = data_slice['sig'].shift(1).fillna(0)
    data_slice['ret'] = data_slice['hourly_return'] * data_slice['pos']

    # Apply costs
    data_slice['exec'] = (data_slice['pos'] != data_slice['pos'].shift(1)).astype(int)
    data_slice['net_ret'] = data_slice['ret'] - (data_slice['exec'] * COST_PER_TRADE)
    return (1 + data_slice['net_ret']).prod() - 1

# Split data into 3 chronological chunks manually to keep DataFrame type
num_chunks = 3
chunk_size = len(final_df) // num_chunks
results = []

for i in range(num_chunks):
    start_idx = i * chunk_size
    # For the last chunk, take all remaining rows
    end_idx = (i + 1) * chunk_size if i < num_chunks - 1 else len(final_df)

    chunk = final_df.iloc[start_idx:end_idx]

    split_idx = int(len(chunk) * 0.7)
    insample = chunk.iloc[:split_idx]
    outsample = chunk.iloc[split_idx:]

    # Find best percentile threshold in-sample
    best_t = 0
    best_r = -np.inf
    for p in [0.5, 0.6, 0.7, 0.8]:
        val = insample['dynamic_smoothing_factor'].quantile(p)
        r = run_backtest(insample, val)
        if r > best_r:
            best_r = r
            best_t = val

    # Test on out-of-sample
    oos_return = run_backtest(outsample, best_t)
    is_return = run_backtest(insample, best_t)
    results.append({'Window': i+1, 'Best_Threshold': best_t, 'IS_Net_Return': is_return , 'OOS_Net_Return': oos_return})

display(pd.DataFrame(results))

,Window,Best_Threshold,IS_Net_Return,OOS_Net_Return
0,1,0.331188,0.073805,-0.075128
1,2,0.137375,0.052368,-0.017769
2,3,0.183186,0.288201,0.018244
